# 03 — BiGRU : comprendre, entraîner et comparer

Nous entraînons depuis zéro un petit réseau de classification sur les mêmes tâches que les baselines : sentiment tunisien **POS/NEG** et variété **MSA/TN/EG**. Contributions partielles à **L04, L07 et L12**.

**Pourquoi cette référence ?** Le cahier des charges demande de comparer des approches classiques, une baseline neuronale légère et des modèles préentraînés adaptés. La BiGRU reste dans les résultats et le rapport même si TF-IDF est meilleur. Elle ne sera pas transformée en Transformer ou en LLM : ces modèles seront expérimentés séparément.

Le réglage par défaut est **`full`** : un nouveau smoke vérifié, puis tout le train et toute la validation, avec **trois graines par tâche**. Exécuter toutes les cellules relance réellement les six apprentissages. Aucun test officiel n'est sélectionné.

Pour relire les résultats existants sans apprendre de nouveau, choisir **`replay`** dans la première cellule. **`smoke`** produit seulement deux petits contrôles techniques ; **`off`** n'exécute rien.

Utiliser le noyau **TuniScope (notebooks)**. Les calculs PyTorch se déroulent dans l'environnement séparé `environments/bigru/.venv/`, via les scripts du projet ; le noyau n'a pas besoin d'importer PyTorch. Prérequis : `sh scripts/bigru_uv.sh sync --locked`, corpus autorisés locaux et manifestes v2.


In [1]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/bigru_uv.sh').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir le notebook depuis le dépôt ou son dossier notebooks/.')
sys.path.insert(0, str(ROOT / 'scripts'))
from bigru_notebook_support import train_bigru, replay_bigru, summary, figures

TRAINING_MODE = os.environ.get('TUNISCOPE_BIGRU_MODE', 'full')
REPLAY_RUN = ROOT / 'outputs/neural_baselines/2026-11-04_bigru_v1_cpu_full'
if TRAINING_MODE not in {'off', 'replay', 'smoke', 'full'}:
    raise ValueError('Choisir off, replay, smoke ou full.')
print('Mode effectif :', TRAINING_MODE.upper())
if 'TUNISCOPE_BIGRU_MODE' in os.environ:
    print('Mode défini par la variable TUNISCOPE_BIGRU_MODE.')
if TRAINING_MODE == 'full':
    print('Nouveau train complet : deux tâches × trois graines ; CPU selon la configuration.')


Mode effectif : FULL
Nouveau train complet : deux tâches × trois graines ; CPU selon la configuration.


## 1. Que reçoit le réseau ?

Le texte est découpé en mots/suites alphanumériques et symboles. Le vocabulaire est appris **uniquement sur le train** ; les unités rares ou hors vocabulaire deviennent **UNK**. Le padding **PAD** complète les lots sans être traité comme du texte.

La séquence est limitée à 128 tokens ; les troncatures sont enregistrées, sans retirer les textes des scores. L'audit initial observe une ligne tronquée en validation TSAC et aucune en validation MADAR. Il ne faut pas confondre tokens UNK avec mots inconnus de la langue.

## 2. Quelles sont ses couches ?

| Étape | Rôle |
| --- | --- |
| Embedding appris, 64 valeurs par token | Table `identifiant → vecteur`, ajustée sur le train |
| GRU bidirectionnelle, 64 valeurs par sens | Lit la séquence dans l'ordre et en sens inverse |
| Moyenne masquée, 128 valeurs par texte | Résume les positions réelles, hors PAD |
| Dropout pendant le train | Désactive aléatoirement une partie des valeurs pour régulariser |
| Couche linéaire | Produit deux scores POS/NEG ou trois scores MSA/TN/EG |

Une **BiGRU** est une GRU bidirectionnelle ; une GRU est un réseau récurrent qui met à jour un état à mesure qu'il lit les tokens. Ce réseau produit un label, pas une réponse rédigée.

## 3. Comment apprend-il ?

Une **époque** parcourt tout le train. Pour chaque lot : prédiction, calcul de la perte, calcul des gradients et mise à jour avec Adam. La validation est ensuite évaluée avec le dropout désactivé et **sans mise à jour des poids**.

Au maximum 12 époques, arrêt après trois sans hausse stricte du macro-F1 de validation ; égalités au premier checkpoint. Les graines **13, 42 et 2026** modifient l'initialisation, l'ordre des lots et le dropout ; elles ne changent pas les partitions. Nous présentons leur moyenne et leur écart-type, sans choisir la meilleure graine.


In [2]:
final_run = None
figure_report = None
if TRAINING_MODE == 'off':
    print('Exécution désactivée : aucun corpus ni modèle chargé, aucun entraînement.')
elif TRAINING_MODE == 'replay':
    final_run = replay_bigru(REPLAY_RUN)
else:
    final_run = train_bigru(TRAINING_MODE)
if final_run is not None:
    print()
    print(summary(final_run))


Contrôle préalable — 100 exemples par partition, deux tâches, graine 42.
BiGRU — smoke — device=cpu — seeds=[42]
TSAC seed=42 epoch=1/2 train_loss=0.7011 val_loss=0.6896 val_macro_f1=0.391172 (0.06s train)
TSAC seed=42 epoch=2/2 train_loss=0.6921 val_loss=0.6916 val_macro_f1=0.345399 (0.07s train)
MADAR seed=42 epoch=1/2 train_loss=1.1341 val_loss=1.0628 val_macro_f1=0.325490 (0.02s train)
MADAR seed=42 epoch=2/2 train_loss=1.0514 val_loss=1.0347 val_macro_f1=0.216216 (0.02s train)

Entraînement FULL — tout le train, toute la validation, trois graines par tâche.
Verified smoke: 2 BiGRU checkpoints, train-only vocabularies, full validation replay and metrics
BiGRU — full — device=cpu — seeds=[13, 42, 2026]
TSAC seed=13 epoch=1/12 train_loss=0.6119 val_loss=0.5332 val_macro_f1=0.719393 (3.72s train)
TSAC seed=13 epoch=2/12 train_loss=0.4535 val_loss=0.4382 val_macro_f1=0.784054 (3.88s train)
TSAC seed=13 epoch=3/12 train_loss=0.3493 val_loss=0.3980 val_macro_f1=0.808209 (3.87s train)
TSA

## 4. Où sont les résultats et les images ?

Chaque entraînement conserve ses modèles, vocabulaires, labels, historiques d'époques et prédictions dans un **nouveau dossier privé**. Aucun ancien run n'est remplacé. Le tableau ci-dessus indique le mode réel et les effectifs ; un résultat smoke n'est jamais présenté comme un full.

Le code suivant utilise **ce même run sélectionné**. Il exporte ses PNG dans `figures/`, ou réutilise un export existant après contrôle des empreintes. Le rechargement du checkpoint et le recalcul des scores font partie de la vérification.


In [2]:
if final_run is not None:
    figure_report = figures(final_run)
    print('Mode des résultats :', figure_report['mode'].upper())
    print('Dossier des modèles et JSON :', final_run['directory'])
    print('Dossier des PNG :', figure_report['directory'])
    print('Images vérifiées :', len(figure_report['images']))


NameError: name 'final_run' is not defined

In [3]:
if figure_report is not None:
    from IPython.display import display, Image
    for filename in figure_report['images']:
        display(Image(filename=str(figure_report['directory'] / filename)))
    print('Terminé —', final_run['mode'].upper(), '—', len(final_run['results']),
          'modèles et', len(figure_report['images']), 'PNG vérifiés.')
    print('Dossier des PNG :', figure_report['directory'])


NameError: name 'figure_report' is not defined

## 5. Comment lire les courbes ?

- **Perte train / validation** : erreur de la fonction d'apprentissage. Une perte train qui baisse tandis que la perte validation remonte peut signaler du surapprentissage.
- **Macro-F1 validation** : qualité des labels avec le même poids pour chaque classe ; le trait vertical indique l'époque retenue selon ce critère.
- **Matrice de confusion** : labels réels en lignes, prédictions en colonnes, pour le checkpoint retenu, qui peut précéder la dernière époque.

La perte et le macro-F1 peuvent évoluer différemment. Le protocole choisit l'époque selon le macro-F1, pas selon la dernière perte observée. Les scores softmax ne sont pas calibrés.

## 6. Quel lien avec la suite du projet ?

| Approche | Ce que nous comparons |
| --- | --- |
| TF-IDF + régression logistique | Référence classique de classification |
| BiGRU apprise depuis zéro | Référence neuronale légère sur notre train |
| Encodeur Transformer préentraîné, puis adapté | Autre système spécialisé pour les mêmes labels |
| Petit LLM avant/après LoRA | Effet de l'adaptation du même LLM sur les instructions de classification |

Les poids de la BiGRU ne servent pas à initialiser l'encodeur ou le LLM. Nous réutilisons les partitions, métriques, contrôles et méthode d'expérimentation. La comparaison entre architectures ne prouve pas à elle seule un effet causal du préentraînement.

Sur l'expérience initiale vérifiée, TF-IDF caractères reste supérieur à la BiGRU : macro-F1 TSAC 0,928898 contre 0,853201 en moyenne ; MADAR 0,949244 contre 0,899806. Un nouveau run doit être interprété selon ses propres sorties ci-dessus. Un résultat moins performant reste une preuve utile et doit être conservé.

La BiGRU n'est pas automatiquement intégrée à l'application. Elle reste un modèle de comparaison possible ; le backend sert actuellement les modèles classiques retenus. **L04/L07/L12 restent partiels** : relectures humaines, autres phases et preuves finales restent ouvertes.

Documentation : [protocole](../docs/38_protocole_bigru.md), [environnement et commandes](../docs/39_environnement_bigru.md), [résultats initiaux](../docs/40_resultats_bigru.md).
